In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
prod_cat = spark.read.format("parquet").load("abfss://bronze@intechaccstorage.dfs.core.windows.net/ProductCategory")
prod_cat.limit(5).display()

ProductCategoryID,ParentProductCategoryID,Name,rowguid,ModifiedDate,_rescued_data
1,null,Bikes,cfbda25c-df71-47a7-b81b-64ee161aa37c,2002-06-01 00:00:00.0000000,null
2,null,Components,c657828d-d808-4aba-91a3-af2ce02300e9,2002-06-01 00:00:00.0000000,null
3,null,Clothing,10a7c342-ca82-48d4-8a38-46a2eb089b74,2002-06-01 00:00:00.0000000,null
4,null,Accessories,2be3be36-d9a2-4eee-b593-ed895d97c2a6,2002-06-01 00:00:00.0000000,null
5,1,Mountain Bikes,2d364ade-264a-433c-b092-4fcbf3804e01,2002-06-01 00:00:00.0000000,null


In [0]:
prod_cat = prod_cat.withColumnRenamed('ProductCategoryID', 'Product_category_id')\
                    .withColumnRenamed('ParentProductCategoryID', 'Parent_product_category_id')\
                    .withColumnRenamed('ModifiedDate', 'Modified_date')\
                    .withColumnRenamed('Name', 'Parent_category_name')
                    
                  
prod_cat = prod_cat.withColumn('Modified_date', to_date(col("Modified_date")))
prod_cat = prod_cat.drop('rowguid', '_rescued_data')

In [0]:
prod_cat.columns

['Product_category_id',
 'Parent_product_category_id',
 'Parent_category_name',
 'Modified_date']

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksintech.silver.product_category
(
    Product_category_id STRING,
    Parent_product_category_id STRING,
    Parent_category_name STRING,
    Modified_date DATE
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/product_category';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.product_category")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = prod_cat.alias("source"),
    condition = (
        "target.Product_category_id = source.Product_category_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
